In [ ]:
!pip install transformers

In [ ]:
!pip install 'transformers[torch]'

In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, T5ForConditionalGeneration, TrainingArguments

In [ ]:
train_data = pd.read_csv('samsum-train.csv')
val_data = pd.read_csv('samsum-validation.csv')

In [ ]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [ ]:
train_data['dialogue'][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [ ]:
train_data.sample(5)

,id,dialogue,summary
12598,13829782,Anna: Are you planning to go to the library to...,Anna will go with Mike to the library tomorrow.
9203,13862246,"Andrew: Hi Peter, my son is going to Lille nex...",Andrew's son is going to Lille next September ...
11093,13730645,Samuel: What's with these French protests? I d...,Samuel is getting scary about the French prote...
11374,13728845,Josh: I'm inviting you for dinner\r\nDon: Toda...,Josh invites Don for dinner. He will announce ...
9266,13611552,Sarah: Mom can I stay at Dom's tonight?\r\nLin...,Linda forbids her daughter Sarah to stay at Do...


In [ ]:
train_data.shape

(14732, 3)

In [ ]:
val_data.shape

(818, 3)

In [ ]:
# random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [ ]:
train_data.shape

(4000, 3)

# Data Preprocessing

In [ ]:
import re

def clean_data(text):
  text = re.sub(r"\r\n"," ", text) # lines
  text = re.sub(r"\r\s+"," ", text) # spaces
  text = re.sub(r"\r<.*?>"," ", text) # html tag
  text = text.strip().lower()
  return text

In [ ]:
train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary'] = train_data['summary'].apply(clean_data)

val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary'] = val_data['summary'].apply(clean_data)

In [ ]:
train_data['dialogue'][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet: <file_other> claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

# Tokenize

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# raw data => tokenized inputs for fine tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length= 512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length= 150, truncation=True)

    inputs["labels"] = targets["input_ids"] # token ids => add to input as labels
    return inputs

In [ ]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset =   val_data.apply(tokenize, axis=1).tolist()

In [ ]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 2, 11966, 834, 9269, 3155, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
# input ids - dialogue => token ids
# 1 => End of sequence , 0 => padding
# attention mask
# labels - target => summary

In [ ]:
len(train_dataset[0]['input_ids'])

512

In [ ]:
type(train_dataset)

list

In [ ]:
type(val_dataset)

list

# Working with our Model

In [ ]:
# NLP => Generation Task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
import torch

# Check whether GPU is available or not
if torch.cuda.is_available():
    device = torch.device("cuda")   # Use NVIDIA GPU
    print("GPU is available!")
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")    # Fallback to CPU
    print("GPU not available, using CPU")

# Move model to selected device (GPU or CPU)
model.to(device)

print("Using device:", device)

GPU is available!
GPU Name: Tesla T4
Using device: cuda


# Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy= "epoch",
    save_strategy= "epoch",
    warmup_steps=500,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
# train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.586243,0.379426
2,0.395991,0.359138
3,0.373225,0.353596
4,0.361157,0.349729
5,0.354601,0.348575
6,0.349701,0.348565


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9034861450195313, metrics={'train_runtime': 1279.8416, 'train_samples_per_second': 18.752, 'train_steps_per_second': 2.344, 'total_flos': 3248203235328000.0, 'train_loss': 0.9034861450195313, 'epoch': 6.0})

In [ ]:
# model load => fine-tune => save the model

In [ ]:
model.save_pretrained('./saved_summary_model')
tokenizer.save_pretrained('./saved_summary_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [ ]:
model = T5ForConditionalGeneration.from_pretrained('./saved_summary_model')
tokenizer = T5Tokenizer.from_pretrained('./saved_summary_model')

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

# Test the core logic for summarization

In [ ]:
def summarize_diaglogue(dialogue):
  diaglogue = clean_data(dialogue) # clean

  # tokenize
  inputs = tokenizer(
      dialogue,
      max_length=512,
      padding="max_length",
      truncation=True,
      return_tensors="pt"
  )

  # generate the summary => token ids
  model.to(device)
  summary_ids = model.generate(
      inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length=150,
      num_beams=4,
      early_stopping=True
  )

  # token ids convert to summary => decoding ( decode our output)
  summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
  return summary

In [ ]:
test_dialogue = """"

"""

summary = summarize_diaglogue(test_dialogue)
print("summary: ", summary)


In [ ]:
from google.colab import files

files.download('saved_summary_model/config.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Zip the folder
!zip -r saved_summary_model.zip saved_summary_model

  adding: saved_summary_model/ (stored 0%)
  adding: saved_summary_model/model.safetensors (deflated 10%)
  adding: saved_summary_model/generation_config.json (deflated 29%)
  adding: saved_summary_model/config.json (deflated 63%)
  adding: saved_summary_model/tokenizer_config.json (deflated 83%)
  adding: saved_summary_model/tokenizer.json (deflated 79%)
